# Geolocalización y Análisis de Establecimientos Sanitarios (REFES) - Argentina

# 📘 1. Introducción y carga del dataset

# Este notebook realiza la geolocalización y análisis territorial de los establecimientos sanitarios registrados en Argentina.
# Fuente de datos: Ministerio de Salud - Registro Federal de Establecimientos de Salud (REFES) https://datos.gob.ar/ja/dataset/salud_336cf4d9-447a-44c4-8e34-0ba1fc293d55/archivo/salud_5d5710df-cf3f-4d91-b9c5-aecab1e06018 

**Nombre:** Listado de Establecimientos Asistenciales Asentados – REFES 
- **Organismo:** Ministerio de Salud de la Nación 
- **Fecha de descarga:** 27/12/2024 
- **Descripción:** Base nacional de establecimientos públicos y privados, con datos como dirección, localidad, provincia, tipo, nivel de complejidad, etc. 

## ⚙️ Tecnologías utilizadas 
 
- Python 3.x 
- Pandas, NumPy 
- Geopy (Nominatim y Google Maps API) 
- Folium (mapas interactivos) 
- Matplotlib / Seaborn 
- Power BI (para visualización avanzada) 
- Jupyter Notebook 
- Microsoft Fabric (opcional para integración empresarial) 
 
--- 
 
## 📁 Estructura del Proyecto 
 
  

📦 PROJ005_Geolocalizacion_REFES 

 │ 

 ├── 01_INPUTS/ # Dataset original .xlsx/.csv 

 ├── 02_DOCUMENTACION/ # README, fuentes, referencias 

 ├── 03_SCRIPTS/ # Scripts .py individuales por fase 

 ├── 04_OUTPUTS/ 

 │ ├── ARCHIVOS_INTERMEDIOS/ # CSV geocodificados, errores 

 │ ├── MAPAS/ # HTML interactivos 

 ├── 05_VISUALIZACIONES/ # Reportes Power BI, mapas 

 ├── geolocalizacion_refes.ipynb # Jupyter Notebook principal 

 ├── requirements.txt 

 └── README.md 

# 📁 1. Crear estructura del proyecto

Creamos la estructura base de carpetas necesarias para almacenar los insumos, salidas intermedias y resultados finales del proyecto.


In [ ]:
# crear_estructura.py

import os

# Ruta base del proyecto
BASE_DIR = r"C:\Users\...\PROJECTS\PROJ005"

# Lista completa de carpetas a crear
folders = [
    "01_DOCUMENTACION",
    "02_INPUTS",
    os.path.join("03_SCRIPTS"),
    os.path.join("04_OUTPUTS", "GRAFICOS"),
    os.path.join("04_OUTPUTS", "RESUMENES"),
    os.path.join("04_OUTPUTS", "ARCHIVOS_INTERMEDIOS"),
    os.path.join("04_OUTPUTS", "GEODATASETS"),
    "05_MAPAS",
    "06_AUTOMATIZACION"
]

# Crear todas las carpetas necesarias
for folder in folders:
    path = os.path.join(BASE_DIR, folder)
    os.makedirs(path, exist_ok=True)
    print(f"📁 Carpeta creada (o ya existía): {path}")

# Crear README inicial
readme_path = os.path.join(BASE_DIR, "README.md")
if not os.path.exists(readme_path):
    with open(readme_path, "w", encoding="utf-8") as f:
        f.write("# Proyecto PROJ005 – Geolocalización Nacional de Establecimientos de Salud\n")
        f.write("Documentación técnica y estructura base generada automáticamente.\n")
    print(f"📝 Archivo creado: {readme_path}")
else:
    print("📝 Archivo README.md ya existe.")

print("\n✅ Estructura completa del proyecto PROJ005 generada correctamente.")


# 📊 2. Carga de datos y generación de 'UBICACION_SUGERIDA'

Se importa el dataset del Registro Federal de Establecimientos de Salud (REFES), y se construye la columna `UBICACION_SUGERIDA` a partir de los campos relevantes para la geolocalización.


In [ ]:
# eda_refes.py

import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# === CONFIGURACIÓN DE RUTAS ===
BASE_DIR = r"C:\Users\...\PROJECTS\PROJ005"
INPUT_FILE = os.path.join(BASE_DIR, "02_INPUTS", "establecimientos-asistenciales-asentados-registro-federal-refes-20241227.xlsx")
OUTPUT_GRAPH = os.path.join(BASE_DIR, "04_OUTPUTS", "GRAFICOS")
OUTPUT_SUMMARY = os.path.join(BASE_DIR, "04_OUTPUTS", "RESUMENES")
OUTPUT_INTERMEDIOS = os.path.join(BASE_DIR, "04_OUTPUTS", "ARCHIVOS_INTERMEDIOS")

# === CREACIÓN DE CARPETAS NECESARIAS ===
for folder in [OUTPUT_GRAPH, OUTPUT_SUMMARY, OUTPUT_INTERMEDIOS]:
    os.makedirs(folder, exist_ok=True)

# === CARGA DEL ARCHIVO ===
df = pd.read_excel(INPUT_FILE)

# === CONFIGURACIÓN DE GRÁFICOS ===
sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

# === GENERAR CAMPO DE UBICACIÓN ===
df["UBICACION_SUGERIDA"] = df[["DOMICILIO", "LOCALIDAD_NOMBRE", "PROVINCIA_NOMBRE"]].fillna("").agg(", ".join, axis=1)
df["UBICACION_INCOMPLETA"] = df[["DOMICILIO", "LOCALIDAD_NOMBRE", "PROVINCIA_NOMBRE"]].isnull().any(axis=1)

# === DISTRIBUCIÓN POR PROVINCIA ===
prov_counts = df["PROVINCIA_NOMBRE"].value_counts()
prov_counts.plot(kind='barh', title="Establecimientos por Provincia")
plt.xlabel("Cantidad")
plt.ylabel("Provincia")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_GRAPH, "provincia_distribucion.png"))
plt.clf()

# === TOP 20 LOCALIDADES ===
loc_counts = df["LOCALIDAD_NOMBRE"].value_counts().head(20)
sns.barplot(x=loc_counts.values, y=loc_counts.index, palette="viridis")
plt.title("Top 20 Localidades con más Establecimientos")
plt.xlabel("Cantidad")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_GRAPH, "localidad_top20.png"))
plt.clf()

# === TIPOS DE ESTABLECIMIENTOS MÁS FRECUENTES ===
tipo_counts = df["TIPOLOGIA_NOMBRE"].value_counts().head(15)
sns.barplot(x=tipo_counts.values, y=tipo_counts.index, palette="magma")
plt.title("Tipos de Establecimientos más frecuentes")
plt.xlabel("Cantidad")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_GRAPH, "tipologia_top15.png"))
plt.clf()

# === RESUMEN DE COLUMNAS ===
with open(os.path.join(OUTPUT_SUMMARY, "resumen_columnas.txt"), "w", encoding="utf-8") as f:
    for col in df.columns:
        f.write(f"{col} → Tipo: {df[col].dtype}, Únicos: {df[col].nunique()}, Nulos: {df[col].isnull().sum()}\n")

# === GUARDAR ARCHIVO CON UBICACIÓN COMPLETA ===
df.to_csv(os.path.join(OUTPUT_INTERMEDIOS, "establecimientos_con_ubicacion.csv"), index=False, encoding="utf-8")

# === GUARDAR ARCHIVO DE REGISTROS A REVISAR ===
df[df["UBICACION_INCOMPLETA"]].to_csv(
    os.path.join(OUTPUT_INTERMEDIOS, "establecimientos_para_revisar.csv"),
    index=False,
    encoding="utf-8"
)

# === CONFIRMACIÓN FINAL ===
print("✅ EDA completado.")
print(f"- Gráficos en: {OUTPUT_GRAPH}")
print(f"- Resumen en: {OUTPUT_SUMMARY}")
print(f"- CSV completo: {OUTPUT_INTERMEDIOS}\\establecimientos_con_ubicacion.csv")
print(f"- A revisar: {OUTPUT_INTERMEDIOS}\\establecimientos_para_revisar.csv")

## 🧹 Paso 3 – Normalización de direcciones

Este paso tiene como objetivo limpiar y estandarizar los textos de la columna `UBICACION_SUGERIDA`, reemplazando abreviaturas y errores comunes que dificultan la geocodificación. Algunos ejemplos incluyen la sustitución de “Av.” por “Avenida” o “Gral.” por “General”.

Se trabaja sobre el mismo archivo exportado tras el paso 2.


In [ ]:
import pandas as pd
import os
import re

# === RUTAS ===
BASE_DIR = r"C:\Users\...\PROJECTS\PROJ005"
INPUT_CSV = os.path.join(BASE_DIR, "04_OUTPUTS", "ARCHIVOS_INTERMEDIOS", "refes_geocodificados_parcial_38767.csv")
INPUT_XLSX = os.path.join(BASE_DIR, "02_INPUTS", "establecimientos-asistenciales-asentados-registro-federal-refes-20241227.xlsx")
OUTPUT_DIR = os.path.join(BASE_DIR, "04_OUTPUTS", "ARCHIVOS_INTERMEDIOS")
OUTPUT_CSV = os.path.join(OUTPUT_DIR, "refes_geocodificados_parcial_38767_normalizado.csv")
OUTPUT_XLSX = os.path.join(OUTPUT_DIR, "establecimientos_asistenciales_refes20241227_normalizado.xlsx")

# === FUNCIONES DE NORMALIZACIÓN ===

def limpiar_texto(texto):
    if pd.isna(texto):
        return ""
    texto = str(texto).upper()
    texto = texto.encode("latin1", errors="ignore").decode("utf-8", errors="ignore")
    texto = re.sub(r"[^\w\s,.:/-]", "", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

def expandir_abreviaturas(ubicacion):
    reemplazos = {
        "AV.": "AVENIDA", "AV ": "AVENIDA ",
        "GRAL.": "GENERAL", "GRAL ": "GENERAL ",
        "STA.": "SANTA", "STO.": "SANTO",
        "PTE.": "PRESIDENTE", "PTE ": "PRESIDENTE ",
        "TTE.": "TENIENTE", "TTE ": "TENIENTE ",
        "INT.": "INTENDENTE", "INT ": "INTENDENTE ",
        "CAP.": "CAPITAN", "CAP ": "CAPITAN ",
        "FRAGATA ": "", "EX ": "", "EX-": "", "EX LOTE": "",
        "NO ": "", "NO N": "",
        "N°": " ", " Nº": " ", " N ": " ", "S/N": "SN", "S/N°": "SN",
        "E/": "ENTRE ", " PTO ": " PUERTO ", "PTO ": "PUERTO ",
        "B°": "BARRIO ", "Bº": "BARRIO ", "BÂº": "BARRIO ",
        "/": " ", " PB": "", " PISO": "", " PLANTA BAJA": "",
        " Y ": " ", " DENTRO DE ": " "
    }
    for k, v in reemplazos.items():
        ubicacion = ubicacion.replace(k, v)
    return ubicacion

def corregir_errores_ortograficos(ubicacion):
    errores = {
        "CORDOB": "CORDOBA", "CÓRDOBA": "CORDOBA",
        "NEUQUN": "NEUQUEN", "NEUQN": "NEUQUEN", "NEUQUÉN": "NEUQUEN",
        "TUCUMN": "TUCUMAN", "TUCUMÁN": "TUCUMAN",
        "ENTRE ROS": "ENTRE RIOS", "ENTRE RÍOS": "ENTRE RIOS",
        "PERÓN": "PERON", "PERN": "PERON",
        "CAMACU": "CAMACUA", "PINZN": "PINZON", "AGERO": "AGÜERO", "MAIP": "MAIPU",
        "JOS": "JOSE", "HERNND": "HERNANDEZ", "OROO": "OROÑO", "GALVN": "GALVAN",
        "JUNN": "JUNIN", "SOLS": "SOLIS", "NUEZ": "NUÑEZ", "MUIZ": "MUÑIZ",
        "GUAYRA": "GUAYRA", "DAZ": "DIAZ", "NGEL": "ANGEL", "LVAREZ": "ALVAREZ",
        "URIBURU": "URIBURU", "SAENZ PEA": "SAENZ PEÑA", "CRMER": "CRAMER",
        "CERVIO": "CERVIÑO", "MONTAESES": "MONTAÑESES", "AVIL": "AVILA", "GIRALDES": "GÜIRALDES",
        "DPT": "DEPARTAMENTO", "DEPTO": "DEPARTAMENTO", "DEPT": "DEPARTAMENTO",
        "B°": "BARRIO", "LOCALIDAD ": "", "LOTE ": "", "PARAJE ": "", "PUESTO ": ""
    }
    for k, v in errores.items():
        if k in ubicacion:
            ubicacion = ubicacion.replace(k, v)
    return ubicacion

def simplificar_alturas(ubicacion):
    observaciones = []
    match = re.search(r"\b(\d{3,5})[\s/]+(\d{2,5})\b", ubicacion)
    if match:
        ubicacion = ubicacion.replace(match.group(0), match.group(1))
        observaciones.append("Doble altura eliminada")
    if " Y " in ubicacion:
        if re.search(r"\d{3,5} Y \d{3,5}", ubicacion):
            ubicacion = re.sub(r"\b(\d{3,5}) Y \d{3,5}\b", r"\1", ubicacion)
            observaciones.append("Doble altura con 'Y' eliminada")
        elif re.search(r"\b[A-Z]+ Y [A-Z]+", ubicacion):
            observaciones.append("Contiene esquina (Y entre calles)")
    return ubicacion, observaciones

def extraer_lat_lon(ubicacion):
    match = re.search(r"LAT[\s:=-]*(-?\d+[.,]\d+)[\s,;]*LON[\s:=-]*(-?\d+[.,]\d+)", ubicacion)
    if match:
        lat = match.group(1).replace(",", ".")
        lon = match.group(2).replace(",", ".")
        return f"LATLON_EXTRAIDO:{lat},{lon}"
    return ubicacion

def normalizar_fila_con_observaciones(fila):
    observaciones = []
    original = fila.get("UBICACION_SUGERIDA", "") or fila.get("DOMICILIO", "")
    ubic = limpiar_texto(original)
    ubic = expandir_abreviaturas(ubic)
    ubic = corregir_errores_ortograficos(ubic)
    ubic, obs_altura = simplificar_alturas(ubic)
    observaciones.extend(obs_altura)
    ubic = extraer_lat_lon(ubic)
    if "DENTRO DE" in ubic:
        ubic = ubic.split("DENTRO DE")[0].strip()
        observaciones.append("Fragmento 'DENTRO DE...' eliminado")
    ubic = re.sub(r" +", " ", ubic).strip()  # Evitar espacios dobles finales
    return pd.Series([original, ubic, "; ".join(observaciones)])

# === CARGA Y PROCESAMIENTO ===

df_csv = pd.read_csv(INPUT_CSV, encoding="utf-8")
df_xlsx = pd.read_excel(INPUT_XLSX)
df_refes = df_xlsx[["ESTABLECIMIENTO_ID", "DOMICILIO"]].rename(columns={"DOMICILIO": "DOMICILIO_REGISTRO_REFES"})

df_csv = df_csv.merge(df_refes, on="ESTABLECIMIENTO_ID", how="left")
df_csv["UBICACION_SUGERIDA"] = df_csv["UBICACION_SUGERIDA"].fillna(df_csv["DOMICILIO_REGISTRO_REFES"])

df_csv[["UBICACION_ORIGINAL", "UBICACION_SUGERIDA", "OBSERVACIONES"]] = df_csv.apply(normalizar_fila_con_observaciones, axis=1)

df_csv.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")

df_xlsx[["UBICACION_ORIGINAL", "UBICACION_SUGERIDA", "OBSERVACIONES"]] = df_xlsx.apply(normalizar_fila_con_observaciones, axis=1)
df_xlsx.to_excel(OUTPUT_XLSX, index=False, engine="openpyxl")

# === RESUMEN ===
registros_con_observaciones = df_csv["OBSERVACIONES"].astype(bool).sum()
print(f"✅ Archivos normalizados y exportados correctamente.")
print(f"ℹ️ Registros con observaciones: {registros_con_observaciones} / {len(df_csv)}")



## 🌍 Paso 4 – Geocodificación con Nominatim (OpenStreetMap)

Se realiza la geolocalización utilizando la API de Nominatim, aplicando `RateLimiter` para evitar el bloqueo por exceso de consultas.  
Se generan checkpoints automáticos cada 2.000 registros, y se guarda el estado de geocodificación en las columnas `latitude`, `longitude` y `geo_status`.


In [ ]:
import pandas as pd
import os
import re
from datetime import datetime
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
from geopy.exc import GeocoderTimedOut, GeocoderServiceError

# === MARCAR INICIO DE EJECUCIÓN ===
inicio = datetime.now()

# === RUTAS ===
BASE_DIR = r"C:\Users\...\PROJECTS\PROJ005"
INPUT_CSV = os.path.join(BASE_DIR, "04_OUTPUTS", "ARCHIVOS_INTERMEDIOS", "establecimientos_con_ubicacion_normalizada.csv")
OUTPUT_FULL = os.path.join(BASE_DIR, "04_OUTPUTS", "GEODATASETS", "refes_latlong.csv")
OUTPUT_ERRORS = os.path.join(BASE_DIR, "04_OUTPUTS", "GEODATASETS", "refes_no_geocodificados.csv")
OUTPUT_CHECKPOINT_DIR = os.path.join(BASE_DIR, "04_OUTPUTS", "ARCHIVOS_INTERMEDIOS")
LOG_FILE = os.path.join(BASE_DIR, "04_OUTPUTS", "RESUMENES", "log_ejecucion.txt")

# === CREAR CARPETAS SI NO EXISTEN ===
os.makedirs(OUTPUT_CHECKPOINT_DIR, exist_ok=True)
os.makedirs(os.path.dirname(LOG_FILE), exist_ok=True)

# === CARGA DEL ARCHIVO ORIGINAL COMPLETO ===
df_total = pd.read_csv(INPUT_CSV)

# === VERIFICACIÓN DE COLUMNA CLAVE ===
if "UBICACION_SUGERIDA" not in df_total.columns:
    print("❌ ERROR: Falta la columna 'UBICACION_SUGERIDA'. Abortando.")
    exit()

# === DETECTAR CHECKPOINT MÁS RECIENTE ===
def get_ultimo_checkpoint():
    pattern = re.compile(r"refes_geocodificados_parcial_(\d+)\.csv")
    archivos = os.listdir(OUTPUT_CHECKPOINT_DIR)
    parciales = [(int(match.group(1)), fname) for fname in archivos if (match := pattern.match(fname))]
    return max(parciales, default=(0, None))

ultimo_idx, archivo_parcial = get_ultimo_checkpoint()

if ultimo_idx > 0:
    df = pd.read_csv(os.path.join(OUTPUT_CHECKPOINT_DIR, archivo_parcial))
    print(f"🔁 Reanudando desde fila {ultimo_idx + 1}...")
else:
    df = df_total.copy()
    df["latitude"] = None
    df["longitude"] = None
    df["geo_status"] = ""
    ultimo_idx = 0
    print("🚀 Iniciando desde el principio...")

# === CONFIGURACIÓN DEL GEOCODIFICADOR ===
geolocator = Nominatim(user_agent="geo_olmos_resume", timeout=10)
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1, max_retries=2, error_wait_seconds=10)

# === PROCESO DE GEOLOCALIZACIÓN CON FALLBACK ===
for idx in range(ultimo_idx, len(df)):
    ubicacion = df.at[idx, "UBICACION_SUGERIDA"]
    fallback = df.at[idx, "ESTABLECIMIENTO_NOMBRE"] if "ESTABLECIMIENTO_NOMBRE" in df.columns else ""
    direccion_final = ubicacion
    print(f"⏳ Geocodificando: {idx + 1}/{len(df)}")

    try:
        if isinstance(ubicacion, str) and ubicacion.strip():
            location = geocode(direccion_final)
            if location:
                df.at[idx, "latitude"] = location.latitude
                df.at[idx, "longitude"] = location.longitude
                df.at[idx, "geo_status"] = "OK"
            else:
                # Intentar con nombre del establecimiento si existe
                if isinstance(fallback, str) and fallback.strip():
                    direccion_con_nombre = f"{fallback}, {ubicacion}"
                    location_fallback = geocode(direccion_con_nombre)
                    if location_fallback:
                        df.at[idx, "latitude"] = location_fallback.latitude
                        df.at[idx, "longitude"] = location_fallback.longitude
                        df.at[idx, "geo_status"] = "OK_CON_NOMBRE"
                    else:
                        df.at[idx, "geo_status"] = "NO_ENCONTRADO"
                else:
                    df.at[idx, "geo_status"] = "NO_ENCONTRADO"
        else:
            df.at[idx, "geo_status"] = "VACIO"
    except GeocoderTimedOut:
        df.at[idx, "geo_status"] = "TIMEOUT"
    except GeocoderServiceError as e:
        df.at[idx, "geo_status"] = f"SERVICE_ERROR: {str(e)}"
    except Exception as e:
        df.at[idx, "geo_status"] = f"ERROR: {str(e)}"

    # === CHECKPOINT CADA 2000 ===
    if (idx + 1) % 2000 == 0 or (idx + 1) == len(df):
        parcial_file = os.path.join(OUTPUT_CHECKPOINT_DIR, f"refes_geocodificados_parcial_{idx+1}.csv")
        df.to_csv(parcial_file, index=False, encoding="utf-8")
        print(f"💾 Guardado parcial: {parcial_file}")

# === GUARDAR RESULTADO COMPLETO ===
df.to_csv(OUTPUT_FULL, index=False, encoding="utf-8")
df[df["geo_status"] != "OK"].to_csv(OUTPUT_ERRORS, index=False, encoding="utf-8")

# === LOG FINAL DE TIEMPO ===
fin = datetime.now()
duracion = fin - inicio
with open(LOG_FILE, "a", encoding="utf-8") as f:
    f.write(f"\n🕒 Inicio: {inicio.strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"🕔 Fin:    {fin.strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"⏱️ Duración: {str(duracion)}\n")
    f.write("="*60 + "\n")

# === RESUMEN FINAL ===
total = len(df)
ok = df["geo_status"].value_counts().get("OK", 0)
ok_nombre = df["geo_status"].value_counts().get("OK_CON_NOMBRE", 0)
errores = total - ok - ok_nombre

print("\n✅ Geocodificación completada.")
print(f"📄 CSV final: {OUTPUT_FULL}")
print(f"✅ Coordenadas OK: {ok}")
print(f"✅ Coordenadas con nombre: {ok_nombre}")
print(f"⚠️ Casos con error: {errores}")
print(f"📝 Log guardado en: {LOG_FILE}")



## 🔁 Paso 5 – Reintentos para geolocalización fallida

Este script detecta registros cuyo `geo_status` indica error o ausencia de coordenadas, y realiza un nuevo intento de geolocalización con una dirección alternativa simplificada.  
También genera checkpoints cada 2.000 registros dentro de la carpeta `REINTENTOS/`.


In [ ]:
import pandas as pd
import os
import re
import time
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError

# === CONFIGURACIÓN ===
INPUT_DIR = r"C:\Users\...\PROJECTS\REINTENTAR"
OUTPUT_DIR = r"C:\Users\...\PROJECTS\PROJ005\04_OUTPUTS\ARCHIVOS_INTERMEDIOS\REINTENTOS"
os.makedirs(OUTPUT_DIR, exist_ok=True)

geolocator = Nominatim(user_agent="geo_olmos_retry", timeout=60)

def geocode_retry(ubicacion, max_retries=5, delay=15):
    for intento in range(max_retries):
        try:
            return geolocator.geocode(ubicacion)
        except (GeocoderTimedOut, GeocoderServiceError):
            print(f"⏳ Reintentando [{intento+1}/{max_retries}] para: {ubicacion}")
            time.sleep(delay * (intento + 1))
        except Exception as e:
            print(f"❌ Error no manejado: {e}")
            return None
    return None

# === FUNCIONES AUXILIARES ===
def limpiar_ubicacion(texto):
    if not isinstance(texto, str):
        return ""
    texto = texto.strip()
    texto = re.sub(r"\s{2,}", " ", texto)
    texto = texto.replace(" ,", ",").replace(", ", ", ")
    return texto

def mejorar_semantica(est_nombre, ubicacion):
    est_nombre = str(est_nombre).upper()
    ubicacion = ubicacion.upper() if isinstance(ubicacion, str) else ""

    if "PUESTO" in est_nombre and "PUESTO" not in ubicacion:
        ubicacion += ", Puesto Sanitario"
    if "CLÍNICA" in est_nombre and "CLÍNICA" not in ubicacion:
        ubicacion += ", Clínica"
    if "MATERNIDAD" in est_nombre and "MATERNIDAD" not in ubicacion:
        ubicacion += ", Maternidad"
    if "BARRIO" in est_nombre and "BARRIO" not in ubicacion:
        partes = est_nombre.split("BARRIO")
        if len(partes) > 1:
            barrio = partes[1].strip()
            if barrio and barrio not in ubicacion:
                ubicacion += f", Barrio {barrio}"
    if "NODO" in est_nombre and "NODO" not in ubicacion:
        nodo = est_nombre.split("NODO")[-1].strip()
        if nodo and nodo not in ubicacion:
            ubicacion += f", Nodo {nodo}"
    return ubicacion.title()

barrio_caba_keywords = [
    "Agronomía", "Almagro", "Balvanera", "Barracas", "Belgrano", "Boedo", "Caballito",
    "Chacarita", "Coghlan", "Colegiales", "Constitución", "Flores", "Floresta", "La Boca",
    "Liniers", "Mataderos", "Monte Castro", "Monserrat", "Nueva Pompeya", "Núñez",
    "Palermo", "Parque Avellaneda", "Parque Chacabuco", "Parque Chas", "Parque Patricios",
    "Puerto Madero", "Recoleta", "Retiro", "Saavedra", "San Cristóbal", "San Nicolás",
    "San Telmo", "Vélez Sársfield", "Versalles", "Villa Crespo", "Villa del Parque",
    "Villa Devoto", "Villa General Mitre", "Villa Lugano", "Villa Luro", "Villa Ortúzar",
    "Villa Pueyrredón", "Villa Real", "Villa Riachuelo", "Villa Santa Rita",
    "Villa Soldati", "Villa Urquiza"
]

def contiene_barrio_caba(ubicacion):
    if not isinstance(ubicacion, str):
        return False
    for barrio in barrio_caba_keywords:
        if barrio.lower() in ubicacion.lower():
            return True
    return False

def forzar_caba_si_corresponde(ubicacion):
    if contiene_barrio_caba(ubicacion):
        return f"{ubicacion}, Ciudad Autónoma de Buenos Aires, Argentina"
    return ubicacion

# === PROCESAMIENTO ORDENADO DE LOS ARCHIVOS orden_1.csv a orden_9.csv ===
for i in range(1, 10):
    filename = f"orden_{i}.csv"
    path_csv = os.path.join(INPUT_DIR, filename)

    if not os.path.exists(path_csv):
        print(f"⚠️ Archivo no encontrado: {path_csv}")
        continue

    print(f"\n📂 Procesando archivo: {filename}")
    df = pd.read_csv(path_csv, encoding="utf-8")

    errores_validos = ["", None, "NO_ENCONTRADO", "TIMEOUT", "VACIO"]
    errores_adicionales = df["geo_status"].fillna("").str.upper().str.startswith(("ERROR", "SERVICE_ERROR"))
    pendientes = df[df["geo_status"].isin(errores_validos) | errores_adicionales].copy()

    if pendientes.empty:
        print("✅ Sin registros pendientes.")
        continue

    contador_ok = 0
    contador_fallos = 0
    total = len(pendientes)

    for j, idx in enumerate(pendientes.index, 1):
        est = pendientes.at[idx, "ESTABLECIMIENTO_NOMBRE"]
        ubicacion = pendientes.at[idx, "UBICACION_SUGERIDA"]
        ubicacion = limpiar_ubicacion(ubicacion)
        ubicacion = mejorar_semantica(est, ubicacion)
        ubicacion = forzar_caba_si_corresponde(ubicacion)

        print(f"📍 [{j}/{total}] Buscando: {ubicacion}")

        if ubicacion:
            location = geocode_retry(ubicacion, max_retries=5, delay=15)
            if location:
                df.at[idx, "latitude"] = location.latitude
                df.at[idx, "longitude"] = location.longitude
                df.at[idx, "geo_status"] = "OK"
                contador_ok += 1
                print(f"✅ ({location.latitude}, {location.longitude})")
            else:
                df.at[idx, "geo_status"] = "NO_ENCONTRADO"
                contador_fallos += 1
                print("⚠️ No encontrado.")
        else:
            df.at[idx, "geo_status"] = "VACIO"
            contador_fallos += 1
            print("⚠️ Dirección vacía.")

        # Checkpoint cada 2000 o último
        if j % 2000 == 0 or j == total:
            checkpoint_name = f"orden_{i}_checkpoint_{j}.csv"
            df.to_csv(os.path.join(OUTPUT_DIR, checkpoint_name), index=False, encoding="utf-8")
            print(f"💾 Checkpoint guardado: {checkpoint_name}")

    df.to_csv(os.path.join(OUTPUT_DIR, f"orden_{i}.csv"), index=False, encoding="utf-8")
    print(f"\n🟢 Finalizado orden_{i}.csv | ✔️ Éxitos: {contador_ok} | ❌ Fallos: {contador_fallos}")



## 📊 Paso 6 – Análisis de resultados de reintentos

Este paso revisa los archivos CSV generados en la carpeta `REINTENTOS/` tras los reintentos de geolocalización.  
Resume la cantidad de registros procesados, los casos exitosos y los errores, y exporta un informe en Excel con los detalles por archivo y estado.


In [ ]:
import pandas as pd
import os

# === RUTAS ===
BASE_DIR = r"C:\Users\...\PROJECTS\PROJ005"
INPUT_FINAL = os.path.join(BASE_DIR, "04_OUTPUTS", "GEODATASETS", "refes_latlong.csv")
INPUT_PREVIO = os.path.join(BASE_DIR, "04_OUTPUTS", "GEODATASETS", "refes_latlong_backup_pre_reintento.csv")
SALIDA_TXT = os.path.join(BASE_DIR, "04_OUTPUTS", "ANALISIS", "comparativa_reintentos.txt")

os.makedirs(os.path.dirname(SALIDA_TXT), exist_ok=True)

# === CARGA ===
print("📥 Cargando resultados previos y posteriores al reintento...")
df_pre = pd.read_csv(INPUT_PREVIO, encoding="utf-8")
df_post = pd.read_csv(INPUT_FINAL, encoding="utf-8")

# === UNIFICACIÓN POR IDENTIFICADOR ===
if "ID" in df_pre.columns:
    clave = "ID"
elif "CUIT" in df_pre.columns:
    clave = "CUIT"
else:
    clave = "ESTABLECIMIENTO_NOMBRE"

# === MÉTRICAS DE ÉXITO ===
ok_pre = df_pre[df_pre["geo_status"] == "OK"].shape[0]
ok_post = df_post[df_post["geo_status"] == "OK"].shape[0]
mejora = ok_post - ok_pre
porcentaje = (mejora / df_pre.shape[0]) * 100

# === ERRORES PERSISTENTES ===
errores_persistentes = df_post[df_post["geo_status"] != "OK"]
top_localidades = errores_persistentes["LOCALIDAD"].value_counts().head(5)

# === EXPORTAR ===
with open(SALIDA_TXT, "w", encoding="utf-8") as f:
    f.write("🔁 COMPARATIVA DE REINTENTOS DE GEOLOCALIZACIÓN\n")
    f.write("="*50 + "\n")
    f.write(f"✔️ Éxitos antes del reintento: {ok_pre}\n")
    f.write(f"✔️ Éxitos después del reintento: {ok_post}\n")
    f.write(f"📈 Mejora absoluta: {mejora} registros\n")
    f.write(f"📊 Mejora porcentual: {porcentaje:.2f}%\n")
    f.write("\n❗Top 5 localidades con errores persistentes:\n")
    for localidad, cant in top_localidades.items():
        f.write(f"  - {localidad}: {cant} errores\n")

print("📄 Comparativa exportada en:", SALIDA_TXT)



## 🔎 Paso 7 – Análisis de errores por tipo y región

Este módulo examina los registros que no pudieron geocodificarse.  
Agrupa por tipo de error, provincia, localidad y nombre del establecimiento para identificar patrones o zonas problemáticas.  
Se genera un archivo de texto con un resumen detallado.


In [ ]:
import pandas as pd
import os

# === RUTAS ===
BASE_DIR = r"C:\Users\...\PROJECTS\PROJ005"
INPUT_FILE = os.path.join(BASE_DIR, "04_OUTPUTS", "GEODATASETS", "refes_latlong.csv")
OUTPUT_TXT = os.path.join(BASE_DIR, "04_OUTPUTS", "ANALISIS", "errores_geocodificacion_resumen.txt")
os.makedirs(os.path.dirname(OUTPUT_TXT), exist_ok=True)

print("📥 Cargando datos para análisis...")
df = pd.read_csv(INPUT_FILE, encoding="utf-8")

df_errores = df[df["geo_status"].fillna("").str.upper().isin(["", "NO_ENCONTRADO", "TIMEOUT", "VACIO"])]

if df_errores.empty:
    print("✅ No hay errores para analizar.")
    exit()

# === FUNCIÓN DE SCORING DE CALIDAD ===
def score_calidad_direccion(ubicacion):
    if not isinstance(ubicacion, str) or ubicacion.strip() == "":
        return 0
    score = 0
    if any(c in ubicacion for c in "0123456789"):
        score += 1
    if "," in ubicacion:
        score += 1
    if len(ubicacion.split()) > 4:
        score += 1
    return score

df_errores["calidad_ubicacion"] = df_errores["UBICACION_SUGERIDA"].apply(score_calidad_direccion)

# === ANÁLISIS ===
resumen = df_errores.groupby(["geo_status", "PROVINCIA", "LOCALIDAD"]).size().reset_index(name="cantidad")
resumen = resumen.sort_values("cantidad", ascending=False)

# === EXPORTAR ===
with open(OUTPUT_TXT, "w", encoding="utf-8") as f:
    f.write("Resumen de errores de geocodificación\n")
    f.write("=" * 40 + "\n")
    for _, row in resumen.iterrows():
        f.write(f"{row['geo_status']} - {row['PROVINCIA']} - {row['LOCALIDAD']}: {row['cantidad']} errores\n")
    f.write("\n\nTop 10 ubicaciones con baja calidad:\n")
    top_baja = df_errores[df_errores["calidad_ubicacion"] <= 1].head(10)
    for i, row in top_baja.iterrows():
        f.write(f"{row['ESTABLECIMIENTO_NOMBRE']} - {row['UBICACION_SUGERIDA']}\n")

print("📄 Análisis exportado en:", OUTPUT_TXT)


## 📤 Paso 8 – Exportación para validación manual

Exporta todos los registros con errores de geolocalización a un archivo Excel, con una hoja por cada tipo de error.  
Este archivo puede utilizarse para validación externa o corrección manual.


In [ ]:
import pandas as pd
import os

# === RUTAS ===
BASE_DIR = r"C:\Users\...\PROJECTS\PROJ005"
INPUT_FILE = os.path.join(BASE_DIR, "04_OUTPUTS", "GEODATASETS", "refes_latlong.csv")
OUTPUT_XLSX = os.path.join(BASE_DIR, "04_OUTPUTS", "VALIDACION_MANUAL", "errores_para_validar.xlsx")
os.makedirs(os.path.dirname(OUTPUT_XLSX), exist_ok=True)

# === CARGA DE DATOS ===
print("📥 Cargando dataset final con geolocalización...")
df = pd.read_csv(INPUT_FILE, encoding="utf-8")

# === FILTRO DE ERRORES ===
df_errores = df[df["geo_status"].fillna("").str.upper() != "OK"].copy()

if df_errores.empty:
    print("✅ No hay errores para exportar. Todos los registros fueron geolocalizados correctamente.")
    exit()

# === CLASIFICACIÓN Y EXPORTACIÓN ===
print(f"📊 Exportando {len(df_errores)} errores para validación manual...")

with pd.ExcelWriter(OUTPUT_XLSX, engine="xlsxwriter") as writer:
    for estado in df_errores["geo_status"].dropna().unique():
        estado_filtrado = df_errores[df_errores["geo_status"] == estado]
        nombre_hoja = str(estado).replace(" ", "_")[:31]  # Límite de Excel
        estado_filtrado.to_excel(writer, sheet_name=nombre_hoja, index=False)

    # Errores sin clasificar (geo_status vacío o nulo)
    errores_vacios = df_errores[df_errores["geo_status"].isnull() | (df_errores["geo_status"].str.strip() == "")]
    if not errores_vacios.empty:
        errores_vacios.to_excel(writer, sheet_name="Sin_Clasificar", index=False)

print(f"📁 Archivo generado: {OUTPUT_XLSX}")


## 🗺️ Paso 9 – Generación de mapa interactivo

En esta sección, se crea un mapa interactivo en HTML utilizando `folium`, con los registros geolocalizados correctamente (`geo_status == "OK"`).  
Se añaden marcadores en la ubicación de cada establecimiento y se guarda el archivo HTML en la carpeta de outputs.


In [ ]:
import pandas as pd
import os
import folium
from folium.plugins import MarkerCluster

# === RUTAS ===
BASE_DIR = r"C:\Users\PROJECTS\PROJ005"
INPUT_CSV = os.path.join(BASE_DIR, "04_OUTPUTS", "ARCHIVOS_INTERMEDIOS", "geocodificados_google_output.csv")
OUTPUT_MAP = os.path.join(BASE_DIR, "05_MAPAS", "mapa_establecimientos.html")
os.makedirs(os.path.dirname(OUTPUT_MAP), exist_ok=True)

# === CARGA Y FILTRADO ===
df = pd.read_csv(INPUT_CSV)
df_ok = df[df["GEOCODE_STATUS"] == "OK"].copy()

if df_ok.empty:
    print("❌ No hay registros con coordenadas válidas para mostrar.")
    exit()

# === MAPA INICIAL CENTRADO EN ARGENTINA ===
mapa = folium.Map(location=[-38.4161, -63.6167], zoom_start=5, tiles="CartoDB positron")

# === CLUSTER DE MARCADORES ===
cluster = MarkerCluster().add_to(mapa)

# === AGREGAR MARCADORES ===
for idx, row in df_ok.iterrows():
    lat = row["LATITUD"]
    lon = row["LONGITUD"]
    nombre = row.get("ESTABLECIMIENTO_NOMBRE", "Sin nombre")
    direccion = row.get("UBICACION_SUGERIDA", "Sin dirección")
    provincia = row.get("PROVINCIA_NOMBRE", "Sin provincia")
    categoria = row.get("CATEGORIA_CORREGIDA", "Sin categoría")

    popup_text = f"<strong>{nombre}</strong><br>{direccion}<br><i>{provincia} - {categoria}</i>"
    folium.Marker(location=[lat, lon], popup=popup_text).add_to(cluster)

# === GUARDAR MAPA ===
mapa.save(OUTPUT_MAP)

print("✅ Mapa generado exitosamente.")
print(f"🗺️ Archivo exportado: {OUTPUT_MAP}")

## ✅ Conclusión

Este proyecto demostró cómo realizar un flujo completo de geolocalización de establecimientos sanitarios utilizando datos públicos del Ministerio de Salud de Argentina (REFES), con apoyo de librerías Python como `geopy`, `pandas` y `folium`.  
Los pasos abordaron desde la limpieza de datos, geocodificación con múltiples fuentes, análisis de errores y visualización final sobre un mapa interactivo.  

Este enfoque es escalable y puede replicarse para otros datasets de infraestructura pública o privada con fines de planificación, investigación o auditoría territorial.
